# Etap 1 - Pozyskanie danych z cBioPortal API

**Projekt:** Analiza przeżywalności pacjentów z glejakami w kohorcie TCGA  
**Autor:** Anna Zimniewicz  
**Data:** 18 maja 2026

## Cel notebooka

Pobranie surowych danych klinicznych i molekularnych dla dwóch zbiorów TCGA z cBioPortal:
- **TCGA-GBM** (Glioblastoma Multiforme, Firehose Legacy) - 619 próbek
- **TCGA-LGG** (Brain Lower Grade Glioma, Firehose Legacy) - 530 próbek

Dane zostaną zapisane jako surowe pliki CSV w folderze `data/raw/`.

## Plan

1. Konfiguracja: import bibliotek, ustawienie bazowego URL API
2. Sprawdzenie statusu serwera cBioPortal (`/api/health`)
3. Pobranie listy wszystkich studies i znalezienie `studyId` dla naszych dwóch kohort
4. Pobranie danych klinicznych dla obu studies
5. Pobranie mutacji dla genów IDH1, IDH2
6. Zapis surowych danych do `../data/raw/`

## Źródło danych

- cBioPortal: https://www.cbioportal.org
- Dokumentacja API: https://www.cbioportal.org/api/swagger-ui/index.html

---

## Sekcja 1 — Konfiguracja

Importy bibliotek, definicja stałych projektu (URL bazowy API, identyfikatory studies, ścieżka do surowych danych).

In [1]:
import requests              
import pandas as pd          
from pathlib import Path    
from datetime import datetime  


print("Wszystkie biblioteki zaimportowane poprawnie.")
print(f"pandas version: {pd.__version__}")
print(f"requests version: {requests.__version__}")

Wszystkie biblioteki zaimportowane poprawnie.
pandas version: 2.3.3
requests version: 2.33.1


In [2]:
# === KONFIGURACJA ===


BASE_URL = "https://www.cbioportal.org/api"


STUDY_IDS = {
    "GBM": "gbm_tcga",       
}


RAW_DATA_DIR = Path("..") / "data" / "raw"


RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


DOWNLOAD_DATE = datetime.now().strftime("%Y-%m-%d")


print(f"Base URL: {BASE_URL}")
print(f"Studies: {STUDY_IDS}")
print(f"Folder docelowy: {RAW_DATA_DIR.resolve()}")  
print(f"Data pobrania: {DOWNLOAD_DATE}")

Base URL: https://www.cbioportal.org/api
Studies: {'GBM': 'gbm_tcga'}
Folder docelowy: C:\Users\anazi\Desktop\praca_dyplomowa\data\raw
Data pobrania: 2026-07-23


---

## Sekcja 2 — Weryfikacja serwera i studies

Sprawdzamy, czy:
- serwer cBioPortal odpowiada poprawnie (`/api/health`),
- studies `gbm_tcga` i `lgg_tcga` istnieją i są to faktycznie TCGA Firehose Legacy,
- liczby pacjentów odpowiadają wartościom widocznym na stronie webowej cBioPortal.

**Uwaga diagnostyczna:** podczas weryfikacji wykryto błąd w polu `allSampleCount` w metadanych API (zwraca `1` zamiast prawidłowej liczby próbek). Bug nie wpływa na dalszą pracę — realną wielkość kohorty ustalamy na podstawie endpointu `/patients`.

In [3]:
# === KROK 1: Sprawdzenie statusu serwera cBioPortal ===


health_url = f"{BASE_URL}/health"


response = requests.get(health_url)


response.raise_for_status()


print(f"URL zapytania: {health_url}")
print(f"Status code: {response.status_code}")
print(f"Czas odpowiedzi: {response.elapsed.total_seconds()} s")
print(f"Treść odpowiedzi: {response.text}")

URL zapytania: https://www.cbioportal.org/api/health
Status code: 200
Czas odpowiedzi: 0.919995 s
Treść odpowiedzi: {"status":"UP"}


In [4]:
# === KROK 2: Pobranie listy wszystkich studies w cBioPortal ===



studies_url = f"{BASE_URL}/studies" 
params = {
    "projection" : "SUMMARY", 
    "pageSize" : 10_000_000,
    "pageNumber" : 0,
    "direction" : "ASC",
}

response = requests.get(studies_url, params=params)
response.raise_for_status()

studies_data = response.json()

print(f"Liczba studies w cBioPortal: {len(studies_data)}")
print(f"Typ obiektu: {type(studies_data)}")
print(f"Typ pierwszego elementu: {type(studies_data[0])}")
print(f"\nKlucze w pierwszym elemencie:")
print(list(studies_data[0].keys()))



Liczba studies w cBioPortal: 538
Typ obiektu: <class 'list'>
Typ pierwszego elementu: <class 'dict'>

Klucze w pierwszym elemencie:
['studyId', 'cancerTypeId', 'name', 'description', 'publicStudy', 'pmid', 'citation', 'groups', 'status', 'importDate', 'allSampleCount', 'referenceGenome', 'readPermission', 'resourceCounts']


In [5]:
# === KROK 3: Konwersja na DataFrame i filtrowanie do naszych studies ===


studies_df = pd.DataFrame(studies_data)

print(f"Wymiary tabeli: {studies_df.shape}") 
print(f"Kolumny: {list(studies_df.columns)}")

print("\nTrzy pierwsze studies (wybrane kolumny):")
display(studies_df[["studyId", "name", "cancerTypeId", "allSampleCount"]].head(3)) #.head(3) zwraca 3 pierwsze wiersze 


Wymiary tabeli: (538, 14)
Kolumny: ['studyId', 'cancerTypeId', 'name', 'description', 'publicStudy', 'pmid', 'citation', 'groups', 'status', 'importDate', 'allSampleCount', 'referenceGenome', 'readPermission', 'resourceCounts']

Trzy pierwsze studies (wybrane kolumny):


,studyId,name,cancerTypeId,allSampleCount
0,biliary_tract_msk_2026,"Hepatobiliary Cancer (MSK, 2026)",biliary_tract,1291
1,acbc_mskcc_2015,"Adenoid Cystic Carcinoma of the Breast (MSK, J...",acbc,12
2,acc_tcga,"Adrenocortical Carcinoma (TCGA, Firehose Legacy)",acc,92


Wyjaśnienie: te trzy wiersze to nowo dodane studies w stanie roboczym ("placeholdery" z 1 testową próbką). Sortowanie alfabetyczne wrzuciło je na początek. Nasze GBM (619 próbek) i LGG (530 próbek) są gdzieś dalej w tabeli.

In [6]:
# === KROK 4: Filtrowanie do naszych dwóch studies (GBM + LGG Firehose Legacy) ===


target_ids = list(STUDY_IDS.values())  # ['gbm_tcga', 'lgg_tcga'] 
print(f"Szukamy studies o ID: {target_ids}")


mask = studies_df["studyId"].isin(target_ids)

our_studies = studies_df[mask]

print(f"\nZnaleziono {len(our_studies)} pasujących studies:")
display(our_studies[["studyId", "name", "cancerTypeId", "allSampleCount", "referenceGenome"]])

Szukamy studies o ID: ['gbm_tcga']

Znaleziono 1 pasujących studies:


,studyId,name,cancerTypeId,allSampleCount,referenceGenome
119,gbm_tcga,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",difg,619,hg19


## Sekcja 3 — Eksploracja kohorty

Pobieramy szczegółowe metadane studies (`projection=DETAILED`) oraz listę pacjentów dla obu kohort. Ustalamy realną wielkość kohorty:

- **TCGA-GBM:** 606 pacjentów
- **TCGA-LGG:** 516 pacjentów
- **Razem:** 1122 pacjentów


In [7]:
# === DIAGNOSTYKA: Sprawdzenie metadanych dla naszych studies z DETAILED projection ===


params_detailed = {
    "projection": "DETAILED",
    "pageSize": 10_000_000,
    "pageNumber": 0,
    "direction": "ASC",
}

response = requests.get(f"{BASE_URL}/studies", params=params_detailed)
response.raise_for_status()
studies_detailed = pd.DataFrame(response.json())

print(f"Liczba kolumn z DETAILED: {len(studies_detailed.columns)}")
print(f"Kolumny: {list(studies_detailed.columns)}")


our_studies_detailed = studies_detailed[studies_detailed["studyId"].isin(target_ids)]

print("\nNasze studies z DETAILED projection:")
display(our_studies_detailed)

Liczba kolumn z DETAILED: 27
Kolumny: ['studyId', 'cancerTypeId', 'name', 'description', 'publicStudy', 'pmid', 'citation', 'groups', 'status', 'importDate', 'allSampleCount', 'sequencedSampleCount', 'cnaSampleCount', 'mrnaRnaSeqSampleCount', 'mrnaRnaSeqV2SampleCount', 'mrnaMicroarraySampleCount', 'miRnaSampleCount', 'methylationHm27SampleCount', 'rppaSampleCount', 'massSpectrometrySampleCount', 'completeSampleCount', 'referenceGenome', 'treatmentCount', 'structuralVariantCount', 'cancerType', 'readPermission', 'resourceCounts']

Nasze studies z DETAILED projection:


,studyId,cancerTypeId,name,description,publicStudy,pmid,citation,groups,status,importDate,...,methylationHm27SampleCount,rppaSampleCount,massSpectrometrySampleCount,completeSampleCount,referenceGenome,treatmentCount,structuralVariantCount,cancerType,readPermission,resourceCounts
247,gbm_tcga,difg,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",TCGA Glioblastoma Multiforme. Source data from...,True,NaN,NaN,PUBLIC,0,2026-01-07 12:59:53,...,285,244,0,136,hg19,0,0,"{'id': 'difg', 'name': 'Diffuse Glioma', 'dedi...",True,[]


## Uwaga diagnostyczna - bug w cBioPortal API

Endpoint `/studies` zwraca `allSampleCount: 1` dla obu naszych studies (`gbm_tcga`, `lgg_tcga`), 
zarówno z `projection=SUMMARY` jak i `DETAILED`. To jest **bug w metadanych cBioPortal** - na stronie 
webowej widać poprawne liczby (619 dla GBM, 530 dla LGG), ale licznik w API się rozjechał.

**Inne pola w DETAILED zawierają prawdziwe liczby per typ danych:**
- `sequencedSampleCount`: 290 (GBM), 286 (LGG) - próbki z danymi mutacji
- `cnaSampleCount`: 577 (GBM), 513 (LGG) - próbki z danymi CNA

**Realna wielkość kohorty zostanie ustalona** na podstawie liczby unikalnych pacjentów 
w pobranych danych klinicznych (krok 5).

In [8]:
# === KROK 5: Pobranie listy pacjentów dla obu studies ===



patients_per_study = {}  # tu będziemy zbierać wyniki

for label, study_id in STUDY_IDS.items():
    
    
  
    url = f"{BASE_URL}/studies/{study_id}/patients"
    
    
    params = {"pageSize": 100_000, "pageNumber": 0}
    response = requests.get(url, params=params)
    response.raise_for_status()
    
   
    patients = pd.DataFrame(response.json())
    
  
    patients_per_study[label] = patients
    
    print(f"{label} ({study_id}): {len(patients)} pacjentów")
    
print(f"\nŁącznie pacjentów: {sum(len(df) for df in patients_per_study.values())}")
print("\nKolumny w DataFrame pacjentów GBM:")
print(list(patients_per_study["GBM"].columns))
print("\nPierwsze 3 wiersze GBM:")
display(patients_per_study["GBM"].head(3))

GBM (gbm_tcga): 606 pacjentów

Łącznie pacjentów: 606

Kolumny w DataFrame pacjentów GBM:
['uniquePatientKey', 'patientId', 'studyId']

Pierwsze 3 wiersze GBM:


,uniquePatientKey,patientId,studyId
0,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga
1,VENHQS0wMi0wMDAzOmdibV90Y2dh,TCGA-02-0003,gbm_tcga
2,VENHQS0wMi0wMDA2OmdibV90Y2dh,TCGA-02-0006,gbm_tcga


---

## Sekcja 4 — Diagnostyka dostępności statusu MGMT i IDH

**Procedura diagnostyczna:**
1. Pobranie atrybutów klinicznych na poziomie pacjenta (`PATIENT`) dla `gbm_tcga` i `lgg_tcga`.
2. Sprawdzenie atrybutów na poziomie próbki (`SAMPLE`).
3. Weryfikacja alternatywnych wersji studies (PanCancer Atlas, GDC 2025, studies publikacyjne).
4. Sprawdzenie surowych plików danych w repozytorium cBioPortal DataHub na GitHubie.

**Konkluzja:** żadne ze sprawdzonych studies cBioPortal nie udostępnia `MGMT_STATUS` i `IDH_STATUS` jako gotowych atrybutów na poziomie pacjenta. Konieczna jest decyzja metodologiczna (patrz Sekcja 5).

In [9]:
# === KROK 6: Pobranie danych klinicznych w formacie LONG ===


clinical_long_per_study = {}

for label, study_id in STUDY_IDS.items():
  
    url = f"{BASE_URL}/studies/{study_id}/clinical-data"
    
   
    params = {
        "clinicalDataType": "PATIENT",
        "projection": "SUMMARY",
        "pageSize": 1_000_000,
        "pageNumber": 0,
    }
    
    print(f"Pobieram dane kliniczne dla {label} ({study_id})...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    
    clinical_long = pd.DataFrame(response.json())
    clinical_long_per_study[label] = clinical_long
    
    print(f"  Pobrano {len(clinical_long):,} wierszy (long format)")
    print(f"  Liczba unikalnych pacjentów: {clinical_long['patientId'].nunique()}")
    print(f"  Liczba unikalnych atrybutów: {clinical_long['clinicalAttributeId'].nunique()}")
    print()


print("Pierwsze 10 wierszy danych klinicznych GBM (format LONG):")
display(clinical_long_per_study["GBM"].head(10))

Pobieram dane kliniczne dla GBM (gbm_tcga)...
  Pobrano 14,830 wierszy (long format)
  Liczba unikalnych pacjentów: 606
  Liczba unikalnych atrybutów: 34

Pierwsze 10 wierszy danych klinicznych GBM (format LONG):


,uniquePatientKey,patientId,studyId,clinicalAttributeId,value
0,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,AGE,44
1,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS,0
2,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DFS_MONTHS,4.5
3,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DFS_STATUS,1:Recurred/Progressed
4,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,ETHNICITY,NOT HISPANIC OR LATINO
5,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,FORM_COMPLETION_DATE,12/16/08
6,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTOLOGICAL_DIAGNOSIS,Untreated primary (de novo) GBM
7,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTORY_LGG_DX_OF_BRAIN_TISSUE,NO
8,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTORY_NEOADJUVANT_TRTYN,Yes
9,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,ICD_10,C71.9


In [10]:
# === KROK 7: Eksploracja - jakie atrybuty kliniczne są dostępne ===


attrs_gbm = set(clinical_long_per_study["GBM"]["clinicalAttributeId"].unique())
attrs_lgg = set(clinical_long_per_study["LGG"]["clinicalAttributeId"].unique())


common_attrs = attrs_gbm & attrs_lgg  


only_gbm = attrs_gbm - attrs_lgg
only_lgg = attrs_lgg - attrs_gbm

print(f"Atrybuty wspólne dla obu studies: {len(common_attrs)}")
print(f"Atrybuty tylko w GBM: {len(only_gbm)}")
print(f"Atrybuty tylko w LGG: {len(only_lgg)}")

print("\n=== Atrybuty wspólne dla obu studies (alfabetycznie) ===")
for attr in sorted(common_attrs):
    print(f"  {attr}")

print("\n=== Atrybuty tylko w LGG ===")
for attr in sorted(only_lgg):
    print(f"  {attr}")

print("\n=== Atrybuty tylko w GBM ===")
for attr in sorted(only_gbm):
    print(f"  {attr}")

KeyError: 'LGG'

In [ ]:
# === DIAGNOZA: sprawdzenie atrybutów na poziomie SAMPLE (próbki) ===


url = f"{BASE_URL}/studies/gbm_tcga/clinical-data"
params = {
    "clinicalDataType": "SAMPLE",   # ← TO jest jedyna zmiana
    "projection": "SUMMARY",
    "pageSize": 1_000_000,
    "pageNumber": 0,
}

response = requests.get(url, params=params)
response.raise_for_status()
sample_clinical_gbm = pd.DataFrame(response.json())

print(f"Pobrano {len(sample_clinical_gbm):,} wierszy danych SAMPLE dla GBM")
print(f"Liczba unikalnych atrybutów SAMPLE: {sample_clinical_gbm['clinicalAttributeId'].nunique()}")


sample_attrs_gbm = sorted(sample_clinical_gbm["clinicalAttributeId"].unique())
print("\n=== Wszystkie atrybuty na poziomie SAMPLE dla GBM ===")
for attr in sample_attrs_gbm:
  
    is_key = any(keyword in attr.upper() for keyword in ["MGMT", "IDH", "METHYL", "MUTATION"])
    marker = "  ⭐ " if is_key else "     "
    print(f"{marker}{attr}")

In [ ]:
# === SYSTEMATYCZNY PRZEGLĄD: jakie studies GBM/LGG są dostępne w cBioPortal ===




mask = studies_df["name"].str.contains("glio", case=False, na=False)
glioma_studies = studies_df[mask][["studyId", "name", "allSampleCount"]]

print(f"Studies dotyczące glejaków: {len(glioma_studies)}")
display(glioma_studies)

In [ ]:
# === DIAGNOSTYKA: PanCancer Atlas studies - sprawdzamy atrybuty PATIENT ===

pancancer_studies = [
    "gbm_tcga_pan_can_atlas_2018",
    "lgg_tcga_pan_can_atlas_2018",
    "difg_tcga_gdc",        # Diffuse Glioma 
    "gbm_tcga_pub2013",     # Brennan 2013
    "gbm_tcga_pub",         # TCGA Nature 2008
]


keywords = ["MGMT", "IDH", "METHYL", "MUTATION_STATUS", "1P", "19Q", "G_CIMP", "GRADE", "SUBTYPE"]

for study_id in pancancer_studies:
    print(f"\n=== {study_id} ===")
    
    url = f"{BASE_URL}/studies/{study_id}/clinical-data"
    params = {
        "clinicalDataType": "PATIENT",
        "projection": "SUMMARY",
        "pageSize": 1_000_000,
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        df = pd.DataFrame(response.json())
        
        n_patients = df["patientId"].nunique()
        attrs = sorted(df["clinicalAttributeId"].unique())
        relevant = [a for a in attrs if any(kw in a.upper() for kw in keywords)]
        
        print(f"  Pacjentów: {n_patients}, atrybutów PATIENT łącznie: {len(attrs)}")
        if relevant:
            print(f"  ⭐ Istotne atrybuty:")
            for a in relevant:
                print(f"     {a}")
        else:
            print(f"  ❌ Brak interesujących atrybutów")
    
    except requests.exceptions.HTTPError as e:
        print(f"  ❌ Błąd HTTP: {e.response.status_code}")

In [ ]:
# === DIAGNOSTYKA: pobranie surowego pliku data_clinical_patient.txt z GitHub DataHub ===

# Bezpośredni URL do surowego pliku z repo cBioPortal/datahub

url = "https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/gbm_tcga/data_clinical_patient.txt"

print(f"Pobieram plik z: {url}")
response = requests.get(url)
print(f"Status: {response.status_code}")
print(f"Rozmiar pliku: {len(response.text):,} znaków")
print(f"\nPierwsze 2000 znaków pliku:\n")
print(response.text[:2000])

---

## Sekcja 5 — Decyzja metodologiczna i plan dalszy

### Podjęta decyzja

Status MGMT i IDH dla pacjentów TCGA-GBM i TCGA-LGG zostaną zaczerpnięte z **Supplementary Materials publikacji Ceccarelli et al. 2016** ("Molecular Profiling Reveals Biologically Discrete Subsets and Pathways of Progression in Diffuse Glioma", *Cell*, 164:550-563). Publikacja zawiera autorytatywną klasyfikację molekularną dla ~1100 pacjentów TCGA, używaną w setkach kolejnych prac.

### Dodatkowa walidacja

Status IDH zostanie dodatkowo zwalidowany przez porównanie z mutacjami genów IDH1 i IDH2 pobranymi z endpointu `/mutations` cBioPortal. Zgodność oczekiwana ~95%+.

### Plan na następną sesję

1. Pobranie danych klinicznych (`/clinical-data`) dla obu studies — wiek, płeć, OS_MONTHS, OS_STATUS, leczenie itd.
2. Pobranie Supplementary Table z publikacji Ceccarelli 2016 (gotowe statusy MGMT, IDH, 1p/19q codeletion, SUBTYPE).
3. Pobranie mutacji IDH1/IDH2 — walidacja statusu IDH.
4. Zapis surowych plików do `data/raw/`.
5. Aktualizacja `README.md` z opisem pobranych danych i datą.



## 6. Wczytanie tabeli klinicznej z Ceccarelli 2016

Pełne dane kliniczne + status molekularny (IDH, MGMT, 1p/19q codeletion) dla całej kohorty 1 122 pacjentów (GBM + LGG) pochodzą z materiałów suplementarnych:

> Ceccarelli M, Barthel FP, Malta TM, et al. *Molecular Profiling Reveals Biologically Discrete Subsets and Pathways of Progression in Diffuse Glioma.* Cell. 2016;164(3):550-563.

Plik `ceccarelli_2016_table_s1.xlsx` zawiera 3 arkusze; interesuje nas tylko **S1A** (TCGA discovery dataset).


In [ ]:
# Ścieżka do pliku 
ceccarelli_path = RAW_DATA_DIR / "ceccarelli_2016_table_s1.xlsx"


ceccarelli = pd.read_excel(
    ceccarelli_path,
    sheet_name="S1A. TCGA discovery dataset",
    header=1
)


print(f"Liczba wierszy:  {len(ceccarelli)}")
print(f"Liczba kolumn:   {len(ceccarelli.columns)}")

In [ ]:
# Pełna lista kolumn 
print("Wszystkie kolumny w tabeli:")
for i, col in enumerate(ceccarelli.columns):
    print(f"  {i:2d}. {col}")
    

In [ ]:
# Podgląd pierwszych 3 wierszy 
kluczowe_kolumny = [
    'Case',
    'Study',
    'Histology',
    'Grade',
    'Age (years at diagnosis)',
    'Gender',
    'Survival (months)',
    'Vital status (1=dead)',
    'IDH status',
    'MGMT promoter status',
    '1p/19q codeletion',
    'IDH/codel subtype',
]

ceccarelli[kluczowe_kolumny].head(3)

## 7. Selekcja kolumn i przygotowanie do mergu

Z 51 kolumn tabeli Ceccarelli wybieramy tylko te, które są nam potrzebne do analizy przeżycia i biostatystyki:

- **Identyfikator i stratyfikacja**: `Case`, `Study`
- **Dane kliniczne**: histologia, grade, wiek, płeć, Karnofsky Performance Score
- **Outcome (analiza przeżycia)**: czas przeżycia, status vitalny 
- **Biomarkery główne**: IDH, MGMT, 1p/19q codeletion, gotowy podtyp IDH/codel
- **Biomarkery dodatkowe** (opcjonalne do dyskusji): TERT, ATRX, mutation count

Przemianowujemy kolumny na **snake_case**

In [ ]:
nazwy_kolumn = {
    # Identyfikatory i stratyfikacja
    'Case': 'patient_id',
    'Study': 'study',
    
    # Dane kliniczne
    'Histology': 'histology',
    'Grade': 'grade',
    'Age (years at diagnosis)': 'age_at_diagnosis',
    'Gender': 'gender',
    'Karnofsky Performance Score': 'kps',
    'Mutation Count': 'mutation_count',
    
    # Outcome - analiza przeżycia
    'Survival (months)': 'os_months',         # OS = Overall Survival
    'Vital status (1=dead)': 'os_event',      # event = zdarzenie (zgon)
    
    # Biomarkery główne
    'IDH status': 'idh_status',
    'MGMT promoter status': 'mgmt_status',
    '1p/19q codeletion': 'codel_1p19q',
    'IDH/codel subtype': 'idh_codel_subtype',
    
    # Biomarkery dodatkowe - do ewentualnej rozszerzonej analizy / dyskusji
    'TERT promoter status': 'tert_promoter_status',
    'ATRX status': 'atrx_status',
}

ceccarelli_clean = ceccarelli[list(nazwy_kolumn.keys())].rename(columns=nazwy_kolumn)

# Sprawdzenie wyniku
print(f"Kolumn przed: 51")
print(f"Kolumn po:    {len(ceccarelli_clean.columns)}")
print()
print("Nowe nazwy kolumn:")
for col in ceccarelli_clean.columns:
    print(f"  - {col}")

In [ ]:
# Podgląd pierwszych 5 wierszy z nowymi nazwami
ceccarelli_clean.head(5)

In [ ]:
ceccarelli_clean.info()

## 8. Walidacja zgodności identyfikatorów Ceccarelli vs cBioPortal



In [ ]:
# Łączymy patientId z obu studiów cBioPortal w jeden zbiór
cbioportal_ids = (
    set(patients_per_study["GBM"]["patientId"])
    | set(patients_per_study["LGG"]["patientId"])
)

# To samo dla Ceccarelli
ceccarelli_ids = set(ceccarelli_clean["patient_id"])

# Porównanie - operatory na zbiorach
common = ceccarelli_ids & cbioportal_ids
only_ceccarelli = ceccarelli_ids - cbioportal_ids
only_cbioportal = cbioportal_ids - ceccarelli_ids

# Raport
print(f"Pacjentów w Ceccarelli:    {len(ceccarelli_ids):>5}")
print(f"Pacjentów w cBioPortal:    {len(cbioportal_ids):>5}")
print(f"Wspólnych (w obu źródłach): {len(common):>5}")
print(f"Tylko w Ceccarelli:         {len(only_ceccarelli):>5}")
print(f"Tylko w cBioPortal:         {len(only_cbioportal):>5}")

if only_ceccarelli:
    print(f"\nPrzykłady ID tylko w Ceccarelli ({len(only_ceccarelli)}):")
    for pid in sorted(only_ceccarelli)[:10]:
        print(f"  - {pid}")

if only_cbioportal:
    print(f"\nPrzykłady ID tylko w cBioPortal ({len(only_cbioportal)}):")
    for pid in sorted(only_cbioportal)[:10]:
        print(f"  - {pid}")

## 9. Zapis tabeli klinicznej do `data/raw/`
Surowy plik xlsx zostaje w `data/raw/` jako referencja

In [ ]:
# Ścieżka pliku wynikowego
ceccarelli_clean_path = RAW_DATA_DIR / "ceccarelli_clinical_clean.csv"

ceccarelli_clean.to_csv(ceccarelli_clean_path, index=False, encoding="utf-8")


if ceccarelli_clean_path.exists():
    rozmiar_kb = ceccarelli_clean_path.stat().st_size / 1024
    print(f"✓ Zapisano: {ceccarelli_clean_path}")
    print(f"  Rozmiar: {rozmiar_kb:.1f} KB")
    print(f"  Wiersze: {len(ceccarelli_clean)}")
    print(f"  Kolumny: {len(ceccarelli_clean.columns)}")
else:
    print("✗ Coś poszło nie tak - plik nie powstał")

## Sekcja 10: Walidacja statusu IDH – uwaga

Próba pobrania danych mutacyjnych IDH1/IDH2 z cBioPortal API zakończyła się błędem 503
(Service Unavailable) w dniu 2026-06-14 – zarówno endpoint `/mutations/fetch`,
jak i `/health` zwróciły 503, co wskazuje na awarię po stronie serwera.

**Decyzja:** Walidacja krzyżowa pominięta. Status IDH z Ceccarelli et al. 2016 (*Cell*, 164:550–563)
przyjęty jako wiarygodne źródło – oznaczenie opierało się na sekwencjonowaniu Sanger + IHC
dla całej kohorty 1122 pacjentów. Walidację można powtórzyć gdy API wróci do działania.